In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
os.environ["PYTHONWARNINGS"] = "ignore"

import sys
import inspect
import itertools
import subprocess
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.patches import Polygon
from matplotlib.colors import Normalize

import celloracle as co
from celloracle.trajectory.oracle_GRN import _do_simulation
from celloracle.trajectory.oracle_utility import _adata_to_df

sc.settings.verbosity = 0
np.random.seed(42)


In [2]:
INPUT_H5AD    = "checkpoint/S06_NKG2C_for_ORACLE.h5ad"
ADJACENCIES   = "checkpoint/S03_adjacencies.csv"
OUT_KO_CSV    = "results/S07_results19_ORACLE_KO_results.csv"
OUT_SYNERGY   = "results/S07_results20_synergy_decomposition.csv"
HEATMAP_PDF   = "plots/S07_plot13_single_pairwise_synergy_heatmap.pdf"
WATERFALL_PDF = "plots/S07_plot14_triple_synergy_waterfall.pdf"

OUT_KO_CSV_HYP    = "results/S07_results21_ORACLE_KO_results_hypoxia.csv"
OUT_SYNERGY_HYP   = "results/S07_results22_synergy_decomposition_hypoxia.csv"
HEATMAP_PDF_HYP   = "plots/S07_plot15_single_pairwise_synergy_heatmap_hypoxia.pdf"
WATERFALL_PDF_HYP = "plots/S07_plot16_triple_synergy_waterfall_hypoxia.pdf"

OUT_H5AD      = "checkpoint/S07_NKG2C_after_ORACLE.h5ad"
SESSION_INFO  = "session/S07_session_info.txt"

NK_CLUSTER      = "NK_cluster_annotated"
EXHAUSTED_LABEL = "Activated Stressed NK"
CYTOTOXIC_LABEL = "Mature Cytotoxic NK"

KO_GENES  = ["SMAD4", "NR4A1", "ATF3"]
HYPOXIA_GATED_GENES = ["NR4A1", "ATF3"]
CONSTITUTIVE_GENES  = ["SMAD4"]
SCENARIOS = {"ideal": 0.0, "realistic": 0.5}

TOP_N_PER_TARGET  = 10
N_PROPAGATION     = 3
FIT_ALPHA         = 10
N_HVG             = 3000

In [3]:
adata = sc.read_h5ad(INPUT_H5AD)
sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor="seurat_v3")
hvg_rank = adata.var[["variances_norm"]].assign(gene=adata.var_names) \
    .sort_values(["variances_norm", "gene"], ascending=[False, True])
top_hvg = set(hvg_rank.index[:N_HVG])
hvg_mask = adata.var_names.isin(top_hvg) | adata.var_names.isin(KO_GENES)
n_before = adata.n_vars
adata = adata[:, hvg_mask].copy()
print(f"Genes: {n_before} -> {adata.n_vars}  |  cells: {adata.n_obs}")

Genes: 14089 -> 3001  |  cells: 6342


In [4]:
oracle = co.Oracle()
oracle.import_anndata_as_raw_count(
    adata=adata,
    cluster_column_name=NK_CLUSTER,
    embedding_name="X_umap",
)
np.random.seed(42)
oracle.perform_PCA()
np.random.seed(42)
oracle.knn_imputation(n_jobs=1)

print("layers:", list(oracle.adata.layers.keys()))

  File "C:\Users\ASUS\miniconda3\envs\oracle\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\ASUS\miniconda3\envs\oracle\lib\subprocess.py", line 503, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\ASUS\miniconda3\envs\oracle\lib\subprocess.py", line 971, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\ASUS\miniconda3\envs\oracle\lib\subprocess.py", line 1456, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,


layers: ['raw_count', 'normalized_count', 'imputed_count']


In [5]:
adjacency_df = pd.read_csv(ADJACENCIES)

adjacency_df = (adjacency_df
                .sort_values("importance", ascending=False)
                .groupby("target", sort=False)
                .head(TOP_N_PER_TARGET))
present_genes = set(oracle.adata.var_names)
adjacency_df = adjacency_df[adjacency_df["TF"].isin(present_genes) & adjacency_df["target"].isin(present_genes)].copy()
print(f"GRN edges (top {TOP_N_PER_TARGET} TFs/target, genes present): {len(adjacency_df):,}  |  TFs: {adjacency_df['TF'].nunique()}")

oracle.TFdict = adjacency_df.groupby("target")["TF"].apply(list).to_dict()
oracle.fit_GRN_for_simulation(GRN_unit="cluster", alpha=FIT_ALPHA)
print(f"GRN fitted (per cluster): {len(oracle.TFdict)} target genes.")

GRN edges (top 10 TFs/target, genes present): 18,769  |  TFs: 291


  0%|          | 0/5 [00:00<?, ?it/s]

GRN fitted (per cluster): 2982 target genes.


In [6]:
X_base = oracle.adata.layers["imputed_count"].astype(np.float32)
is_exhausted = (oracle.adata.obs[NK_CLUSTER] == EXHAUSTED_LABEL).values
is_cytotoxic = (oracle.adata.obs[NK_CLUSTER] == CYTOTOXIC_LABEL).values

def mean_centered_diff(a, b):
    d = a.mean(axis=0) - b.mean(axis=0)
    return d - d.mean()

shift_centered = mean_centered_diff(X_base[is_cytotoxic], X_base[is_exhausted])
shift_norm_sq  = float(np.dot(shift_centered, shift_centered))

baseline_mean = pd.Series(X_base.mean(axis=0), index=oracle.adata.var_names)
print(f"Exhausted: {is_exhausted.sum()}  |  Cytotoxic: {is_cytotoxic.sum()}  |  "
      f"||shift_centered||={np.sqrt(shift_norm_sq):.3f}")

Exhausted: 1057  |  Cytotoxic: 2037  |  ||shift_centered||=3.679


In [7]:
def condition_key(genes, scenario):
    ordered = [gene for gene in KO_GENES if gene in genes]
    return "+".join(ordered) + "_" + scenario

def run_ko(genes, scenario):
    fraction = SCENARIOS[scenario]
    perturb_condition = {gene: float(fraction * baseline_mean[gene]) for gene in genes}
    oracle.simulate_shift(perturb_condition=perturb_condition, n_propagation=N_PROPAGATION, ignore_warning=True)
    X_sim = oracle.adata.layers["simulated_count"].astype(np.float32)
    delta_centered = mean_centered_diff(X_sim[is_exhausted], X_base[is_exhausted])
    projection = float(np.dot(delta_centered, shift_centered) / shift_norm_sq) if shift_norm_sq > 0 else 0.0
    return {"projection": projection, "magnitude": float(np.linalg.norm(delta_centered))}

conditions = []
for scenario in SCENARIOS:
    for combo_size in (1, 2, 3):
        for genes in itertools.combinations(KO_GENES, combo_size):
            conditions.append({"genes": genes, "scenario": scenario, "condition_key": condition_key(genes, scenario)})

print(f"{len(conditions)} conditions to simulate\n")

effects, effect_rows = {}, []
for condition in conditions:
    simulation_result = run_ko(condition["genes"], condition["scenario"])
    effects[condition["condition_key"]] = simulation_result["projection"]
    effect_rows.append({"condition": condition["condition_key"], "genes": "+".join(condition["genes"]), "scenario": condition["scenario"],
                         "n_genes": len(condition["genes"]), "projection": round(simulation_result["projection"], 6),
                         "magnitude": round(simulation_result["magnitude"], 6)})
    print(f"{condition['condition_key']:<30s} projection={simulation_result['projection']:+.4f}")

effects_df = pd.DataFrame(effect_rows)
effects_df.to_csv(OUT_KO_CSV, index=False)

14 conditions to simulate

SMAD4_ideal                    projection=-0.0000
NR4A1_ideal                    projection=+0.0828
ATF3_ideal                     projection=+0.0243
SMAD4+NR4A1_ideal              projection=+0.0828
SMAD4+ATF3_ideal               projection=+0.0243
NR4A1+ATF3_ideal               projection=+0.0853
SMAD4+NR4A1+ATF3_ideal         projection=+0.0853
SMAD4_realistic                projection=-0.0000
NR4A1_realistic                projection=+0.0608
ATF3_realistic                 projection=+0.0177
SMAD4+NR4A1_realistic          projection=+0.0608
SMAD4+ATF3_realistic           projection=+0.0177
NR4A1+ATF3_realistic           projection=+0.0625
SMAD4+NR4A1+ATF3_realistic     projection=+0.0625


In [8]:
SMAD4, NR4A1, ATF3 = KO_GENES

def percent_change(actual, expected):
    return float((actual - expected) / abs(expected) * 100) if expected != 0 else float("nan")

decomposition, decomposition_rows = {}, []
for scenario in SCENARIOS:
    SMAD4_effect = effects[condition_key((SMAD4,), scenario)]
    NR4A1_effect = effects[condition_key((NR4A1,), scenario)]
    ATF3_effect  = effects[condition_key((ATF3,), scenario)]
    SMAD4_NR4A1_effect = effects[condition_key((SMAD4, NR4A1), scenario)]
    SMAD4_ATF3_effect  = effects[condition_key((SMAD4, ATF3), scenario)]
    NR4A1_ATF3_effect  = effects[condition_key((NR4A1, ATF3), scenario)]
    triple_effect       = effects[condition_key((SMAD4, NR4A1, ATF3), scenario)]

    SMAD4_NR4A1_interaction = SMAD4_NR4A1_effect - SMAD4_effect - NR4A1_effect
    SMAD4_ATF3_interaction  = SMAD4_ATF3_effect - SMAD4_effect - ATF3_effect
    NR4A1_ATF3_interaction  = NR4A1_ATF3_effect - NR4A1_effect - ATF3_effect
    pure_triple_interaction = (triple_effect - SMAD4_NR4A1_effect - SMAD4_ATF3_effect - NR4A1_ATF3_effect
                                + SMAD4_effect + NR4A1_effect + ATF3_effect)
    total_synergy = triple_effect - SMAD4_effect - NR4A1_effect - ATF3_effect

    decomposition[scenario] = {
        "singles": {SMAD4: SMAD4_effect, NR4A1: NR4A1_effect, ATF3: ATF3_effect},
        "SMAD4_NR4A1_interaction": SMAD4_NR4A1_interaction,
        "SMAD4_ATF3_interaction": SMAD4_ATF3_interaction,
        "NR4A1_ATF3_interaction": NR4A1_ATF3_interaction,
        "pure_triple_interaction": pure_triple_interaction,
        "total_synergy": total_synergy,
    }

    decomposition_rows += [
        {"scenario": scenario, "term": f"{SMAD4} (single)", "actual_projection": SMAD4_effect, "expected_projection": None, "percent_change": None},
        {"scenario": scenario, "term": f"{NR4A1} (single)", "actual_projection": NR4A1_effect, "expected_projection": None, "percent_change": None},
        {"scenario": scenario, "term": f"{ATF3} (single)", "actual_projection": ATF3_effect, "expected_projection": None, "percent_change": None},
        {"scenario": scenario, "term": f"{SMAD4} x {NR4A1} (pair)", "actual_projection": SMAD4_NR4A1_effect,
         "expected_projection": SMAD4_effect + NR4A1_effect, "percent_change": percent_change(SMAD4_NR4A1_effect, SMAD4_effect + NR4A1_effect)},
        {"scenario": scenario, "term": f"{SMAD4} x {ATF3} (pair)", "actual_projection": SMAD4_ATF3_effect,
         "expected_projection": SMAD4_effect + ATF3_effect, "percent_change": percent_change(SMAD4_ATF3_effect, SMAD4_effect + ATF3_effect)},
        {"scenario": scenario, "term": f"{NR4A1} x {ATF3} (pair)", "actual_projection": NR4A1_ATF3_effect,
         "expected_projection": NR4A1_effect + ATF3_effect, "percent_change": percent_change(NR4A1_ATF3_effect, NR4A1_effect + ATF3_effect)},
        {"scenario": scenario, "term": f"{SMAD4} x {NR4A1} x {ATF3} -- total synergy (vs. 3 singles)",
         "actual_projection": triple_effect, "expected_projection": SMAD4_effect + NR4A1_effect + ATF3_effect,
         "percent_change": percent_change(triple_effect, SMAD4_effect + NR4A1_effect + ATF3_effect)},
        {"scenario": scenario, "term": f"{SMAD4} x {NR4A1} x {ATF3} -- pure triple (vs. 3 pairs)",
         "actual_projection": triple_effect,
         "expected_projection": SMAD4_NR4A1_effect + SMAD4_ATF3_effect + NR4A1_ATF3_effect - (SMAD4_effect + NR4A1_effect + ATF3_effect),
         "percent_change": percent_change(triple_effect, SMAD4_NR4A1_effect + SMAD4_ATF3_effect + NR4A1_ATF3_effect - (SMAD4_effect + NR4A1_effect + ATF3_effect))},
    ]

decomposition_df = pd.DataFrame(decomposition_rows)
decomposition_df.to_csv(OUT_SYNERGY, index=False)
decomposition_df

,scenario,term,actual_projection,expected_projection,percent_change
0,ideal,SMAD4 (single),-0.000009,NaN,NaN
1,ideal,NR4A1 (single),0.082798,NaN,NaN
2,ideal,ATF3 (single),0.024293,NaN,NaN
3,ideal,SMAD4 x NR4A1 (pair),0.082788,0.082789,-0.000182
4,ideal,SMAD4 x ATF3 (pair),0.024284,0.024284,-0.000023
5,ideal,NR4A1 x ATF3 (pair),0.085280,0.107091,-20.366367
6,ideal,SMAD4 x NR4A1 x ATF3 -- total synergy (vs. 3 s...,0.085271,0.107082,-20.368232
7,ideal,SMAD4 x NR4A1 x ATF3 -- pure triple (vs. 3 pairs),0.085271,0.085271,0.000037
8,realistic,SMAD4 (single),-0.000003,NaN,NaN
9,realistic,NR4A1 (single),0.060781,NaN,NaN


In [9]:
gene_row_order = KO_GENES
gene_col_order = KO_GENES
pair_to_interaction_key = {
    frozenset((SMAD4, NR4A1)): "SMAD4_NR4A1_interaction",
    frozenset((SMAD4, ATF3)):  "SMAD4_ATF3_interaction",
    frozenset((NR4A1, ATF3)):  "NR4A1_ATF3_interaction",
}
pair_to_genes = {
    frozenset((SMAD4, NR4A1)): (SMAD4, NR4A1),
    frozenset((SMAD4, ATF3)):  (SMAD4, ATF3),
    frozenset((NR4A1, ATF3)):  (NR4A1, ATF3),
}

norm = Normalize(vmin=0.0, vmax=0.15)
cmap = plt.get_cmap("YlGn")

fig, ax = plt.subplots(figsize=(6.4, 5.4))
for row_idx, gene_row in enumerate(gene_row_order):
    for col_idx, gene_col in enumerate(gene_col_order):
        if gene_row == gene_col:
            ideal_projection     = decomposition["ideal"]["singles"][gene_row]
            realistic_projection = decomposition["realistic"]["singles"][gene_row]
            ax.add_patch(Polygon([(col_idx - .5, row_idx + .5), (col_idx + .5, row_idx + .5), (col_idx - .5, row_idx - .5)],
                                 closed=True, facecolor=cmap(norm(ideal_projection)), edgecolor="white"))
            ax.add_patch(Polygon([(col_idx + .5, row_idx + .5), (col_idx + .5, row_idx - .5), (col_idx - .5, row_idx - .5)],
                                 closed=True, facecolor=cmap(norm(realistic_projection)), edgecolor="white"))
        else:
            scenario = "ideal" if row_idx > col_idx else "realistic"
            gene_pair = frozenset((gene_row, gene_col))
            pair_projection   = effects[condition_key(pair_to_genes[gene_pair], scenario)]
            interaction_value = decomposition[scenario][pair_to_interaction_key[gene_pair]]
            ax.add_patch(Polygon([(col_idx - .5, row_idx - .5), (col_idx + .5, row_idx - .5), (col_idx + .5, row_idx + .5), (col_idx - .5, row_idx + .5)],
                                 closed=True, facecolor=cmap(norm(pair_projection)), edgecolor="white"))
            ax.text(col_idx, row_idx, f"{interaction_value:+.3f}", ha="center", va="center", fontsize=8)

ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-0.5, 2.5)
ax.set_xticks(range(3)); ax.set_xticklabels(gene_col_order)
ax.set_yticks(range(3)); ax.set_yticklabels(gene_row_order)
ax.set_title("Above diagonal = ideal | Below diagonal = realistic", fontsize=9, pad=22)
ax.text(0.5, 1.03, "Colour = projection | Number = synergy/redundancy",
        transform=ax.transAxes, ha="center", va="bottom", fontsize=7, style="italic")
scalar_mappable = cm.ScalarMappable(norm=norm, cmap=cmap); scalar_mappable.set_array([])
fig.colorbar(scalar_mappable, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(HEATMAP_PDF)
plt.show()

In [10]:
bar_width = 0.6
gene_spacing = 0.7
ideal_positions = [i * gene_spacing for i in range(4)]
realistic_positions = [ideal_positions[-1] + 1.2 + i * gene_spacing for i in range(4)]
bar_labels = KO_GENES + ["Total"]
gene_colors = {SMAD4: "#1f77b4", NR4A1: "#2ca02c", ATF3: "#17becf", "total": "#9467bd"}

fig, ax = plt.subplots(figsize=(6.5, 5.5))
for scenario, bar_positions, alpha in [("ideal", ideal_positions, 1.0), ("realistic", realistic_positions, 0.7)]:
    SMAD4_effect  = effects[condition_key((SMAD4,), scenario)]
    NR4A1_effect  = effects[condition_key((NR4A1,), scenario)]
    ATF3_effect   = effects[condition_key((ATF3,), scenario)]
    triple_effect = effects[condition_key((SMAD4, NR4A1, ATF3), scenario)]

    gene_effects = {SMAD4: SMAD4_effect, NR4A1: NR4A1_effect, ATF3: ATF3_effect}
    cumulative_projection = np.concatenate([[0.0], np.cumsum(list(gene_effects.values()))])

    for gene_idx, gene in enumerate(KO_GENES):
        ax.bar(bar_positions[gene_idx], gene_effects[gene], bottom=cumulative_projection[gene_idx],
               width=bar_width, color=gene_colors[gene], alpha=alpha, edgecolor="none")
    ax.bar(bar_positions[3], triple_effect, bottom=0, width=bar_width, color=gene_colors["total"], alpha=alpha, edgecolor="none")

    cluster_center_x = sum(bar_positions) / len(bar_positions)
    ax.text(cluster_center_x, -0.06, scenario.upper(), transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=10)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(ideal_positions + realistic_positions)
ax.set_xticklabels(bar_labels * 2, fontsize=8)
ax.set_ylabel("Projection")

fig.tight_layout()
fig.savefig(WATERFALL_PDF)
plt.show()

In [11]:
oracle.adata.write_h5ad(OUT_H5AD)

with open(SESSION_INFO, "w") as f:
    f.write(f"Python {sys.version}\n\n")
    f.write(subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout)

In [12]:
hypoxia_cols = [c for c in adata.obs.columns if "HYPOX" in c.upper()]
HYPOXIA_COL = hypoxia_cols[0]
hyp_scores = adata.obs[HYPOXIA_COL].values.astype(float)

auc_scaled = (hyp_scores - hyp_scores.min()) / (hyp_scores.max() - hyp_scores.min())
print(f"Hypoxia column: {HYPOXIA_COL}")
print(f"dataset-wide mean: {auc_scaled.mean():.3f}")

print(f"auc_scaled mean, Activated/Stressed NK only: {auc_scaled[is_exhausted].mean():.3f}")

Hypoxia column: HALLMARK_HYPOXIA_auc
dataset-wide mean: 0.500
auc_scaled mean, Activated/Stressed NK only: 0.559


In [13]:
gem_imputed  = _adata_to_df(oracle.adata, "imputed_count")
cluster_info = oracle.adata.obs[oracle.cluster_column_name]
cluster_names = np.unique(cluster_info)

def run_ko_hypoxia(genes, scenario):
    fraction = SCENARIOS[scenario]
    simulation_input = gem_imputed.copy()

    for gene in genes:
        target_value = float(fraction * baseline_mean[gene])
        if gene in HYPOXIA_GATED_GENES:
            baseline_values = gem_imputed[gene].values
            simulation_input[gene] = (
                baseline_values + auc_scaled * (target_value - baseline_values)
            )
        else:
            simulation_input[gene] = target_value

    simulated_parts = []
    for cluster in cluster_names:
        mask = (cluster_info == cluster).values
        simulated_parts.append(_do_simulation(
            coef_matrix=oracle.coef_matrix_per_cluster[cluster],
            simulation_input=simulation_input[mask],
            gem=gem_imputed[mask],
            n_propagation=N_PROPAGATION,
        ))
    gem_simulated = pd.concat(simulated_parts, axis=0).reindex(gem_imputed.index)

    X_sim = gem_simulated.values.astype(np.float32)
    delta_centered = mean_centered_diff(X_sim[is_exhausted], X_base[is_exhausted])
    projection = float(np.dot(delta_centered, shift_centered) / shift_norm_sq) if shift_norm_sq > 0 else 0.0
    return {"projection": projection, "magnitude": float(np.linalg.norm(delta_centered))}

effects_hypoxia, effect_rows_hypoxia = {}, []
for condition in conditions:
    simulation_result = run_ko_hypoxia(condition["genes"], condition["scenario"])
    effects_hypoxia[condition["condition_key"]] = simulation_result["projection"]
    effect_rows_hypoxia.append({"condition": condition["condition_key"], "genes": "+".join(condition["genes"]), "scenario": condition["scenario"],
                         "n_genes": len(condition["genes"]), "projection": round(simulation_result["projection"], 6),
                         "magnitude": round(simulation_result["magnitude"], 6)})
    print(f"{condition['condition_key']:<30s} projection={simulation_result['projection']:+.4f}")

effects_hypoxia_df = pd.DataFrame(effect_rows_hypoxia)
effects_hypoxia_df.to_csv(OUT_KO_CSV_HYP, index=False)


SMAD4_ideal                    projection=-0.0000
NR4A1_ideal                    projection=+0.0523
ATF3_ideal                     projection=+0.0153
SMAD4+NR4A1_ideal              projection=+0.0523
SMAD4+ATF3_ideal               projection=+0.0153
NR4A1+ATF3_ideal               projection=+0.0538
SMAD4+NR4A1+ATF3_ideal         projection=+0.0538
SMAD4_realistic                projection=-0.0000
NR4A1_realistic                projection=+0.0398
ATF3_realistic                 projection=+0.0116
SMAD4+NR4A1_realistic          projection=+0.0398
SMAD4+ATF3_realistic           projection=+0.0116
NR4A1+ATF3_realistic           projection=+0.0410
SMAD4+NR4A1+ATF3_realistic     projection=+0.0410


In [14]:
decomposition_hypoxia, decomposition_hypoxia_rows = {}, []
for scenario in SCENARIOS:
    SMAD4_effect = effects_hypoxia[condition_key((SMAD4,), scenario)]
    NR4A1_effect = effects_hypoxia[condition_key((NR4A1,), scenario)]
    ATF3_effect  = effects_hypoxia[condition_key((ATF3,), scenario)]
    SMAD4_NR4A1_effect = effects_hypoxia[condition_key((SMAD4, NR4A1), scenario)]
    SMAD4_ATF3_effect  = effects_hypoxia[condition_key((SMAD4, ATF3), scenario)]
    NR4A1_ATF3_effect  = effects_hypoxia[condition_key((NR4A1, ATF3), scenario)]
    triple_effect       = effects_hypoxia[condition_key((SMAD4, NR4A1, ATF3), scenario)]

    SMAD4_NR4A1_interaction = SMAD4_NR4A1_effect - SMAD4_effect - NR4A1_effect
    SMAD4_ATF3_interaction  = SMAD4_ATF3_effect - SMAD4_effect - ATF3_effect
    NR4A1_ATF3_interaction  = NR4A1_ATF3_effect - NR4A1_effect - ATF3_effect
    pure_triple_interaction = (triple_effect - SMAD4_NR4A1_effect - SMAD4_ATF3_effect - NR4A1_ATF3_effect
                                + SMAD4_effect + NR4A1_effect + ATF3_effect)
    total_synergy = triple_effect - SMAD4_effect - NR4A1_effect - ATF3_effect

    decomposition_hypoxia[scenario] = {
        "singles": {SMAD4: SMAD4_effect, NR4A1: NR4A1_effect, ATF3: ATF3_effect},
        "SMAD4_NR4A1_interaction": SMAD4_NR4A1_interaction,
        "SMAD4_ATF3_interaction": SMAD4_ATF3_interaction,
        "NR4A1_ATF3_interaction": NR4A1_ATF3_interaction,
        "pure_triple_interaction": pure_triple_interaction,
        "total_synergy": total_synergy,
    }

    decomposition_hypoxia_rows += [
        {"scenario": scenario, "term": f"{SMAD4} (single)", "actual_projection": SMAD4_effect, "expected_projection": None, "percent_change": None},
        {"scenario": scenario, "term": f"{NR4A1} (single)", "actual_projection": NR4A1_effect, "expected_projection": None, "percent_change": None},
        {"scenario": scenario, "term": f"{ATF3} (single)", "actual_projection": ATF3_effect, "expected_projection": None, "percent_change": None},
        {"scenario": scenario, "term": f"{SMAD4} x {NR4A1} (pair)", "actual_projection": SMAD4_NR4A1_effect,
         "expected_projection": SMAD4_effect + NR4A1_effect, "percent_change": percent_change(SMAD4_NR4A1_effect, SMAD4_effect + NR4A1_effect)},
        {"scenario": scenario, "term": f"{SMAD4} x {ATF3} (pair)", "actual_projection": SMAD4_ATF3_effect,
         "expected_projection": SMAD4_effect + ATF3_effect, "percent_change": percent_change(SMAD4_ATF3_effect, SMAD4_effect + ATF3_effect)},
        {"scenario": scenario, "term": f"{NR4A1} x {ATF3} (pair)", "actual_projection": NR4A1_ATF3_effect,
         "expected_projection": NR4A1_effect + ATF3_effect, "percent_change": percent_change(NR4A1_ATF3_effect, NR4A1_effect + ATF3_effect)},
        {"scenario": scenario, "term": f"{SMAD4} x {NR4A1} x {ATF3} -- total synergy (vs. 3 singles)",
         "actual_projection": triple_effect, "expected_projection": SMAD4_effect + NR4A1_effect + ATF3_effect,
         "percent_change": percent_change(triple_effect, SMAD4_effect + NR4A1_effect + ATF3_effect)},
        {"scenario": scenario, "term": f"{SMAD4} x {NR4A1} x {ATF3} -- pure triple (vs. 3 pairs)",
         "actual_projection": triple_effect,
         "expected_projection": SMAD4_NR4A1_effect + SMAD4_ATF3_effect + NR4A1_ATF3_effect - (SMAD4_effect + NR4A1_effect + ATF3_effect),
         "percent_change": percent_change(triple_effect, SMAD4_NR4A1_effect + SMAD4_ATF3_effect + NR4A1_ATF3_effect - (SMAD4_effect + NR4A1_effect + ATF3_effect))},
    ]

decomposition_hypoxia_df = pd.DataFrame(decomposition_hypoxia_rows)
decomposition_hypoxia_df.to_csv(OUT_SYNERGY_HYP, index=False)
decomposition_hypoxia_df


,scenario,term,actual_projection,expected_projection,percent_change
0,ideal,SMAD4 (single),-0.000009,NaN,NaN
1,ideal,NR4A1 (single),0.052276,NaN,NaN
2,ideal,ATF3 (single),0.015295,NaN,NaN
3,ideal,SMAD4 x NR4A1 (pair),0.052266,0.052266,-0.000162
4,ideal,SMAD4 x ATF3 (pair),0.015286,0.015286,-0.000007
5,ideal,NR4A1 x ATF3 (pair),0.053840,0.067571,-20.321168
6,ideal,SMAD4 x NR4A1 x ATF3 -- total synergy (vs. 3 s...,0.053830,0.067562,-20.324078
7,ideal,SMAD4 x NR4A1 x ATF3 -- pure triple (vs. 3 pairs),0.053830,0.053830,-0.000022
8,realistic,SMAD4 (single),-0.000003,NaN,NaN
9,realistic,NR4A1 (single),0.039824,NaN,NaN


In [15]:
norm_hyp = Normalize(vmin=0.0, vmax=0.15)
cmap_hyp = plt.get_cmap("YlGn")

fig, ax = plt.subplots(figsize=(6.4, 5.4))
for row_idx, gene_row in enumerate(gene_row_order):
    for col_idx, gene_col in enumerate(gene_col_order):
        if gene_row == gene_col:
            ideal_projection     = decomposition_hypoxia["ideal"]["singles"][gene_row]
            realistic_projection = decomposition_hypoxia["realistic"]["singles"][gene_row]
            ax.add_patch(Polygon([(col_idx - .5, row_idx + .5), (col_idx + .5, row_idx + .5), (col_idx - .5, row_idx - .5)],
                                 closed=True, facecolor=cmap_hyp(norm_hyp(ideal_projection)), edgecolor="white"))
            ax.add_patch(Polygon([(col_idx + .5, row_idx + .5), (col_idx + .5, row_idx - .5), (col_idx - .5, row_idx - .5)],
                                 closed=True, facecolor=cmap_hyp(norm_hyp(realistic_projection)), edgecolor="white"))
        else:
            scenario = "ideal" if row_idx > col_idx else "realistic"
            gene_pair = frozenset((gene_row, gene_col))
            pair_projection   = effects_hypoxia[condition_key(pair_to_genes[gene_pair], scenario)]
            interaction_value = decomposition_hypoxia[scenario][pair_to_interaction_key[gene_pair]]
            ax.add_patch(Polygon([(col_idx - .5, row_idx - .5), (col_idx + .5, row_idx - .5), (col_idx + .5, row_idx + .5), (col_idx - .5, row_idx + .5)],
                                 closed=True, facecolor=cmap_hyp(norm_hyp(pair_projection)), edgecolor="white"))
            ax.text(col_idx, row_idx, f"{interaction_value:+.3f}", ha="center", va="center", fontsize=8)

ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-0.5, 2.5)
ax.set_xticks(range(3)); ax.set_xticklabels(gene_col_order)
ax.set_yticks(range(3)); ax.set_yticklabels(gene_row_order)
ax.set_title("Hypoxia-regressed | Above diagonal = ideal | Below diagonal = realistic", fontsize=9, pad=22)
ax.text(0.5, 1.03, "Colour = projection | Number = synergy/redundancy",
        transform=ax.transAxes, ha="center", va="bottom", fontsize=7, style="italic")
scalar_mappable = cm.ScalarMappable(norm=norm_hyp, cmap=cmap_hyp); scalar_mappable.set_array([])
fig.colorbar(scalar_mappable, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(HEATMAP_PDF_HYP)
plt.show()

In [16]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for scenario, bar_positions, alpha in [("ideal", ideal_positions, 1.0), ("realistic", realistic_positions, 0.7)]:
    SMAD4_effect  = effects_hypoxia[condition_key((SMAD4,), scenario)]
    NR4A1_effect  = effects_hypoxia[condition_key((NR4A1,), scenario)]
    ATF3_effect   = effects_hypoxia[condition_key((ATF3,), scenario)]
    triple_effect = effects_hypoxia[condition_key((SMAD4, NR4A1, ATF3), scenario)]

    gene_effects = {SMAD4: SMAD4_effect, NR4A1: NR4A1_effect, ATF3: ATF3_effect}
    cumulative_projection = np.concatenate([[0.0], np.cumsum(list(gene_effects.values()))])

    for gene_idx, gene in enumerate(KO_GENES):
        ax.bar(bar_positions[gene_idx], gene_effects[gene], bottom=cumulative_projection[gene_idx],
               width=bar_width, color=gene_colors[gene], alpha=alpha, edgecolor="none")
    ax.bar(bar_positions[3], triple_effect, bottom=0, width=bar_width, color=gene_colors["total"], alpha=alpha, edgecolor="none")

    cluster_center_x = sum(bar_positions) / len(bar_positions)
    ax.text(cluster_center_x, -0.06, scenario.upper(), transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=10)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(ideal_positions + realistic_positions)
ax.set_xticklabels(bar_labels * 2, fontsize=8)
ax.set_ylabel("Projection")
ax.set_title("Hypoxia-regressed", fontsize=10)

fig.tight_layout()
fig.savefig(WATERFALL_PDF_HYP)
plt.show()

In [17]:
# Uniform vs hypoxia-gated projection, condition by condition.
comparison_rows = []
for condition in conditions:
    key = condition["condition_key"]
    uniform_val = effects[key]
    hypoxia_val = effects_hypoxia[key]
    pct = percent_change(hypoxia_val, uniform_val)
    comparison_rows.append({"condition": key, "uniform_projection": round(uniform_val, 6),
                             "hypoxia_projection": round(hypoxia_val, 6), "percent_change": pct})
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv("results/S07_results23_uniform_vs_hypoxia_comparison.csv", index=False)
comparison_df


,condition,uniform_projection,hypoxia_projection,percent_change
0,SMAD4_ideal,-0.000009,-0.000009,0.000000
1,NR4A1_ideal,0.082798,0.052276,-36.863675
2,ATF3_ideal,0.024293,0.015295,-37.038180
3,SMAD4+NR4A1_ideal,0.082788,0.052266,-36.867756
4,SMAD4+ATF3_ideal,0.024284,0.015286,-37.052192
5,NR4A1+ATF3_ideal,0.085280,0.053840,-36.867447
6,SMAD4+NR4A1+ATF3_ideal,0.085271,0.053830,-36.871445
7,SMAD4_realistic,-0.000003,-0.000003,0.000000
8,NR4A1_realistic,0.060781,0.039824,-34.479524
9,ATF3_realistic,0.017678,0.011585,-34.468433
